# IT-FGSM Attack on FraudNet

Adversarial evasion attack — perturbs numerical features to flip fraud predictions to not-fraud.

**Attacker assumptions:** has `best.pt` + `metadata.pkl`, knows the model architecture.

## 1. Config — set your paths and attack parameters here

In [1]:
CHECKPOINT  = "../Model/DL_Notebooks/runs/fraud_nn/best.pt"        # path to saved model checkpoint
METADATA    = "../data/Preprocessed/column_metadata.pkl"   # path to metadata pickle
TEST_DATA   = "../data/Preprocessed/test.csv"  # path to test CSV
TARGET_COL  = "isFraud"        # label column name

# Attack hyperparameters
EPSILON     = 0.1    # max L-inf perturbation budget
ALPHA       = 0.01   # step size per iteration  (good rule: epsilon / steps)
STEPS       = 20     # number of gradient iterations
BATCH_SIZE  = 512

## 2. Imports

In [2]:
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

## 3. Device

In [3]:
device = (
    torch.device("mps")  if torch.backends.mps.is_available() else
    torch.device("cuda") if torch.cuda.is_available()         else
    torch.device("cpu")
)
print(f"Using device: {device}")

Using device: mps


## 4. Model definition

Must match the saved checkpoint exactly.

In [4]:
class FraudNet(nn.Module):
    def __init__(self, cat_cols, vocab_sizes, num_dim, hidden_dim, dropout=0.3):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(vocab_sizes[col], min(50, (vocab_sizes[col] + 1) // 2))
            for col in cat_cols
        ])
        emb_dim   = sum(min(50, (vocab_sizes[col] + 1) // 2) for col in cat_cols)
        input_dim = emb_dim + num_dim
        h1, h2    = hidden_dim, hidden_dim // 2
        h3        = max(h2 // 2, 32)

        def _block(in_f, out_f):
            return nn.Sequential(
                nn.Linear(in_f, out_f), nn.BatchNorm1d(out_f), nn.GELU(), nn.Dropout(p=dropout)
            )

        self.net = nn.Sequential(
            _block(input_dim, h1), _block(h1, h2), _block(h2, h3), nn.Linear(h3, 1)
        )

    def forward(self, cat, num):
        embs = [e(cat[:, i]) for i, e in enumerate(self.embeddings)]
        x    = torch.cat(embs + [num], dim=1)
        return self.net(x).squeeze(1)

## 5. Dataset

In [5]:
class FraudDataset(Dataset):
    def __init__(self, df, cat_cols, num_cols, target):
        self.cat = torch.tensor(df[cat_cols].values, dtype=torch.long)
        self.num = torch.tensor(df[num_cols].values, dtype=torch.float32)
        self.y   = torch.tensor(df[target].values,   dtype=torch.float32)
    def __len__(self):       return len(self.y)
    def __getitem__(self, i): return self.cat[i], self.num[i], self.y[i]

## 6. Load checkpoint and metadata

In [6]:
with open(METADATA, "rb") as f:
    metadata = pickle.load(f)

cat_cols    = metadata["cat_cols"]
num_cols    = metadata["num_cols"]
vocab_sizes = metadata["vocab_sizes"]

ckpt       = torch.load(CHECKPOINT, map_location=device)
hidden_dim = ckpt.get("hidden_dim", 256)
dropout    = ckpt.get("dropout",    0.3)

model = FraudNet(cat_cols, vocab_sizes, len(num_cols), hidden_dim, dropout).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"Loaded: {CHECKPOINT}  |  hidden_dim={hidden_dim}  dropout={dropout}")

Loaded: ../Model/DL_Notebooks/runs/fraud_nn/best.pt  |  hidden_dim=256  dropout=0.3


## 7. Load test data

In [7]:
df       = pd.read_csv(TEST_DATA)
cat_cols = [c for c in cat_cols if c in df.columns]
num_cols = [c for c in num_cols if c in df.columns]

dataset  = FraudDataset(df, cat_cols, num_cols, TARGET_COL)
loader   = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f"Test samples: {len(dataset)}")

Test samples: 88581


## 8. IT-FGSM attack function

In [8]:
def itfgsm_attack(model, cat, num, epsilon, alpha, steps, device):
    """
    Iterative FGSM — evasion attack on numerical features.
    Goal: decrease fraud logit so prediction flips from 1 → 0.
    """    
    model.eval()
    cat      = cat.to(device)
    adv_num  = num.clone().detach().to(device)
    num_orig = num.clone().detach().to(device)

    for _ in range(steps):
        adv_num.requires_grad_(True)
        loss = model(cat, adv_num).mean()   # mean fraud logit
        model.zero_grad()
        loss.backward()

        with torch.no_grad():
            adv_num = adv_num - alpha * adv_num.grad.sign()          # step toward not-fraud
            delta   = torch.clamp(adv_num - num_orig, -epsilon, epsilon)  # stay in L-inf ball
            adv_num = (num_orig + delta).detach()

    return adv_num

## 9. Run attack and collect results

In [9]:
all_orig_preds, all_adv_preds = [], []
all_orig_probs, all_adv_probs = [], []
fraud_total = flipped = 0

for cat_batch, num_batch, _ in loader:
    with torch.no_grad():
        orig_probs = torch.sigmoid(model(cat_batch.to(device), num_batch.to(device))).cpu()
    orig_preds = (orig_probs >= 0.5).int().numpy()

    adv_num_batch = num_batch.clone()
    fraud_idx     = np.where(orig_preds == 1)[0]
    fraud_total  += len(fraud_idx)

    if len(fraud_idx):
        adv_num_batch[fraud_idx] = itfgsm_attack(
            model, cat_batch[fraud_idx], num_batch[fraud_idx],
            EPSILON, ALPHA, STEPS, device
        ).cpu()

    with torch.no_grad():
        adv_probs = torch.sigmoid(model(cat_batch.to(device), adv_num_batch.to(device))).cpu()
    adv_preds = (adv_probs >= 0.5).int().numpy()

    flipped += int(((orig_preds == 1) & (adv_preds == 0)).sum())
    all_orig_preds.extend(orig_preds.tolist())
    all_adv_preds.extend(adv_preds.tolist())
    all_orig_probs.extend(orig_probs.numpy().tolist())
    all_adv_probs.extend(adv_probs.numpy().tolist())

print("Done.")

Done.


## 10. Results

In [10]:
evasion_rate = 100.0 * flipped / fraud_total if fraud_total else 0.0

print("=" * 50)
print("IT-FGSM ATTACK RESULTS")
print("=" * 50)
print(f"Fraud samples targeted        : {fraud_total}")
print(f"Fraud detected  (clean)       : {int(np.array(all_orig_preds).sum())}")
print(f"Fraud detected  (adversarial) : {int(np.array(all_adv_preds).sum())}")
print(f"Successful evasions (1 → 0)   : {flipped}")
print(f"Evasion rate                  : {evasion_rate:.1f}%")
print("=" * 50)

IT-FGSM ATTACK RESULTS
Fraud samples targeted        : 2650
Fraud detected  (clean)       : 2650
Fraud detected  (adversarial) : 1602
Successful evasions (1 → 0)   : 1048
Evasion rate                  : 39.5%


## 11. Save adversarial results

In [11]:
out = df.copy()
out["orig_pred"] = all_orig_preds
out["adv_pred"]  = all_adv_preds
out["orig_prob"] = all_orig_probs
out["adv_prob"]  = all_adv_probs
out["evaded"]    = ((out["orig_pred"] == 1) & (out["adv_pred"] == 0)).astype(int)
out.to_csv("adversarial_results.csv", index=False)
print("Saved → adversarial_results.csv")

Saved → adversarial_results.csv
